In [5]:
import nbformat

# Load the notebook
with open("Evalautaion of validated data.ipynb", "r") as f:
    nb = nbformat.read(f, as_version=4)

# Fix the metadata.widgets issue
if "widgets" in nb.metadata:
    if "state" not in nb.metadata.widgets:
        nb.metadata.widgets = {"application/vnd.jupyter.widget-state+json": {"state": {}, "version_major": 2, "version_minor": 0}}

# Save the fixed notebook
with open("Evalautaion of validated data.ipynb", "w") as f:
    nbformat.write(nb, f)

print("Fixed!")

FileNotFoundError: [Errno 2] No such file or directory: 'Evalautaion of validated data.ipynb'

In [1]:
!pip install sacrebleu unbabel-comet --break-system-packages

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 68.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Fou

In [1]:
!pip install bert-score datsets -q

ERROR: Could not find a version that satisfies the requirement datsets (from versions: none)
ERROR: No matching distribution found for datsets


In [ ]:
"""
MT Evaluation: chrF, BLEU, AfricoMET
Evaluates English→Hausa translation quality using the validated sheet.

References  = question_hausa_validated  (expert-corrected Hausa)
Hypotheses  = question_hausa_with_options  (system/raw translation)

Usage:
    pip install sacrebleu unbabel-comet --break-system-packages
    python evaluate_mt.py
    python evaluate_mt.py --hyp_col question_hausa_raw   # evaluate raw translation
    python evaluate_mt.py --dataset your-username/hausa-pq-speech-validated  # load from HF
"""

import argparse
import pandas as pd
import sacrebleu
from datasets import load_dataset

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
HF_DATASET   = "honourjesus/hausa-pq-speech-validated"

REF_COL = "question_hausa_validated"   # gold / reference
HYP_COL = "question_hausa_with_options"  # system hypothesis
SRC_COL = "question_english"           # source (for COMET / AfricoMET)
# ─────────────────────────────────────────────


# ── helpers ──────────────────────────────────

def load_data(hf_dataset=None, local_file=LOCAL_FILE):
    if hf_dataset:
        print(f"Loading from HuggingFace: {hf_dataset}")
        ds = load_dataset(hf_dataset, split="train")
        return ds.to_pandas()
    else:
        print(f"Loading from local file: {local_file}")
        df = pd.read_excel(local_file, sheet_name="validated")
        df.columns = [c.strip().replace('"', '') for c in df.columns]
        rename = {
            "question_text_english":                   "question_english",
            "question_text_hausa":                     "question_hausa_raw",
            "added_Option_to_question_text_hausa":     "question_hausa_with_options",
            "validated_question_text_hausa":           "question_hausa_validated",
            "correct_answer_english":                  "answer_english",
            "correct_answer_hausa":                    "answer_hausa",
        }
        df = df.rename(columns=rename)
        return df


def filter_pairs(df, ref_col, hyp_col, src_col):
    """Keep only rows where both reference and hypothesis exist."""
    mask = df[ref_col].notna() & df[hyp_col].notna()
    if src_col:
        mask = mask & df[src_col].notna()
    df_eval = df[mask].copy()
    print(f"\nRows available for evaluation: {len(df_eval):,}  "
          f"(dropped {len(df) - len(df_eval):,} with missing ref/hyp)")
    return df_eval


# ── metrics ──────────────────────────────────

def compute_bleu(refs, hyps):
    """Corpus-level BLEU (sacrebleu)."""
    result = sacrebleu.corpus_bleu(hyps, [refs])
    return result


def compute_chrf(refs, hyps, word_order=0):
    """
    Corpus-level chrF (sacrebleu).
    word_order=0  → chrF
    word_order=2  → chrF++ (adds word n-gram component)
    """
    result = sacrebleu.corpus_chrf(hyps, [refs], word_order=word_order)
    return result


def compute_africamet(srcs, refs, hyps):
    """
    AfricoMET / AfriCOMET — neural MT metric fine-tuned on African languages.
    Falls back gracefully if model cannot be downloaded.

    Requires: pip install unbabel-comet
    Model:    masakhane/africomet-mtl  (or africamet-qe for reference-free)
    """
    try:
        from comet import download_model, load_from_checkpoint

        print("\nDownloading AfricoMET model (first run may take a few minutes) …")
        model_path = download_model("masakhane/africamet-mtl")
        model = load_from_checkpoint(model_path)

        data = [
            {"src": s, "mt": h, "ref": r}
            for s, h, r in zip(srcs, hyps, refs)
        ]
        output = model.predict(data, batch_size=16, gpus=0)
        scores = output["scores"]
        system_score = output["system_score"]
        return scores, system_score

    except ImportError:
        print("\n⚠  unbabel-comet not installed. Run:")
        print("    pip install unbabel-comet --break-system-packages")
        return None, None
    except Exception as e:
        print(f"\n⚠  AfricoMET failed: {e}")
        return None, None


# ── segment-level breakdown ───────────────────

def segment_level_report(refs, hyps, n=10):
    """Print top-N and bottom-N sentences by chrF score."""
    scores = [
        sacrebleu.sentence_chrf(h, [r]).score
        for r, h in zip(refs, hyps)
    ]
    indexed = sorted(enumerate(scores), key=lambda x: x[1])

    print(f"\n{'─'*60}")
    print(f"SEGMENT-LEVEL chrF  (bottom {n})")
    print(f"{'─'*60}")
    for i, sc in indexed[:n]:
        print(f"  [{i:4d}] chrF={sc:.2f}")
        print(f"         REF: {refs[i][:120]}")
        print(f"         HYP: {hyps[i][:120]}")
        print()

    print(f"\n{'─'*60}")
    print(f"SEGMENT-LEVEL chrF  (top {n})")
    print(f"{'─'*60}")
    for i, sc in indexed[-n:][::-1]:
        print(f"  [{i:4d}] chrF={sc:.2f}")
        print(f"         REF: {refs[i][:120]}")
        print(f"         HYP: {hyps[i][:120]}")
        print()


# ── main ─────────────────────────────────────

def main(args):
    df = load_data(hf_dataset=args.dataset, local_file=LOCAL_FILE)
    df_eval = filter_pairs(df, REF_COL, args.hyp_col, SRC_COL)

    refs = df_eval[REF_COL].tolist()
    hyps = df_eval[args.hyp_col].tolist()
    srcs = df_eval[SRC_COL].tolist() if SRC_COL in df_eval.columns else None

    print(f"\nEvaluating: REF='{REF_COL}'  HYP='{args.hyp_col}'")
    print("=" * 60)

    # ── BLEU ──
    bleu = compute_bleu(refs, hyps)
    print(f"\n📊 BLEU")
    print(f"   Score : {bleu.score:.2f}")
    print(f"   Detail: {bleu}")

    # ── chrF ──
    chrf = compute_chrf(refs, hyps, word_order=0)
    chrfpp = compute_chrf(refs, hyps, word_order=2)
    print(f"\n📊 chrF")
    print(f"   chrF   : {chrf.score:.2f}")
    print(f"   chrF++ : {chrfpp.score:.2f}  (word_order=2)")

    # ── AfricoMET ──
    print(f"\n📊 AfricoMET")
    if srcs:
        seg_scores, sys_score = compute_africamet(srcs, refs, hyps)
        if sys_score is not None:
            print(f"   System score : {sys_score:.4f}")
        else:
            print("   Not computed (see warning above).")
    else:
        print("   Skipped — source column not available.")

    # ── Summary table ──
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"  {'Metric':<15} {'Score':>8}")
    print(f"  {'─'*23}")
    print(f"  {'BLEU':<15} {bleu.score:>8.2f}")
    print(f"  {'chrF':<15} {chrf.score:>8.2f}")
    print(f"  {'chrF++':<15} {chrfpp.score:>8.2f}")
    if srcs and 'sys_score' in dir() and sys_score is not None:
        print(f"  {'AfricoMET':<15} {sys_score:>8.4f}")
    print()

    # ── Segment-level breakdown ──
    if args.segment:
        segment_level_report(refs, hyps, n=args.segment)

    # ── Save scores to CSV ──
    out_df = pd.DataFrame({
        "source": srcs if srcs else [""] * len(refs),
        "reference": refs,
        "hypothesis": hyps,
        "chrf_segment": [
            sacrebleu.sentence_chrf(h, [r]).score
            for r, h in zip(refs, hyps)
        ],
        "bleu_segment": [
            sacrebleu.sentence_bleu(h, [r]).score
            for r, h in zip(refs, hyps)
        ],
    })
    out_path = "/content/mt_evaluation_scores.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Segment-level scores saved → {out_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="MT Evaluation: chrF, BLEU, AfricoMET")
    parser.add_argument(
        "--hyp_col",
        default=HYP_COL,
        help=f"Column to use as hypothesis (default: '{HYP_COL}')",
    )
    parser.add_argument(
        "--dataset",
        default=HF_DATASET,
        help="HuggingFace dataset repo ID to load from (overrides local file)",
    )
    parser.add_argument(
        "--segment",
        type=int,
        default=5,
        help="Print top/bottom N sentences by chrF (default: 5, 0 to skip)",
    )
    args = parser.parse_args()
    main(args)

In [4]:
"""
MT Evaluation: chrF, BLEU, TER, BERTScore, AfricoMET + TTR Analysis
Evaluates English→Hausa translation quality using the validated sheet.
"""

import argparse
import pandas as pd
import sacrebleu
from datasets import load_dataset

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
HF_DATASET = "honourjesus/hausa-pq-speech-validated"

REF_COL = "question_hausa_validated"
HYP_COL = "question_hausa_with_options"
SRC_COL = "question_english"
# ─────────────────────────────────────────────


def load_data(hf_dataset=None, local_file=None):
    if hf_dataset:
        print(f"Loading from HuggingFace: {hf_dataset}")
        ds = load_dataset(hf_dataset, split="train")
        return ds.to_pandas()
    else:
        print(f"Loading from local file: {local_file}")
        df = pd.read_excel(local_file, sheet_name="validated")
        df.columns = [c.strip().replace('"', '') for c in df.columns]
        rename = {
            "question_text_english":               "question_english",
            "question_text_hausa":                 "question_hausa_raw",
            "added_Option_to_question_text_hausa": "question_hausa_with_options",
            "validated_question_text_hausa":       "question_hausa_validated",
            "correct_answer_english":              "answer_english",
            "correct_answer_hausa":                "answer_hausa",
        }
        return df.rename(columns=rename)


def filter_pairs(df, ref_col, hyp_col, src_col):
    mask = df[ref_col].notna() & df[hyp_col].notna()
    if src_col:
        mask = mask & df[src_col].notna()
    df_eval = df[mask].copy()
    print(f"\nRows for evaluation: {len(df_eval):,}  "
          f"(dropped {len(df) - len(df_eval):,} with missing ref/hyp)")
    return df_eval


# ── TTR ──────────────────────────────────────

def compute_ttr(texts, label=""):
    """Type-Token Ratio — lexical diversity measure."""
    all_tokens = []
    for t in texts:
        if isinstance(t, str):
            all_tokens.extend(t.lower().split())
    if not all_tokens:
        return 0.0
    ttr = (len(set(all_tokens)) / len(all_tokens)) * 100
    print(f"   TTR ({label}): {ttr:.4f}  "
          f"[{len(set(all_tokens)):,} types / {len(all_tokens):,} tokens]")
    return ttr


def ttr_analysis(df):
    """Compute TTR across splits matching your friend's Pidgin paper format."""
    print(f"\n{'─'*60}")
    print("TYPE-TOKEN RATIO (TTR) ANALYSIS")
    print(f"{'─'*60}")
    print("Mirrors methodology from Eng-PidginBioData (Adelani et al.)")
    print()

    # Use full dataset for train-like split (no fixed splits defined yet)
    df_valid = df[df[REF_COL].notna()].copy()
    n = len(df_valid)
    train = df_valid.iloc[:int(n * 0.6)]
    dev   = df_valid.iloc[int(n * 0.6):int(n * 0.8)]
    test  = df_valid.iloc[int(n * 0.8):]

    results = {}
    for split_name, split_df in [("Train", train), ("Dev", dev), ("Test", test)]:
        print(f"  [{split_name}]  n={len(split_df)}")
        ttr_en = compute_ttr(split_df[SRC_COL].tolist(), "English")
        ttr_ha = compute_ttr(split_df[REF_COL].tolist(), "Hausa")
        results[split_name] = {"size": len(split_df),
                               "ttr_english": ttr_en,
                               "ttr_hausa": ttr_ha}
        print()

    # Print table
    print(f"\n  {'Split':<8} {'Size':>6}  {'TTR(English)':>13}  {'TTR(Hausa)':>11}")
    print(f"  {'─'*45}")
    for split, v in results.items():
        print(f"  {split:<8} {v['size']:>6}  {v['ttr_english']:>13.4f}  {v['ttr_hausa']:>11.4f}")
    print()
    print("  Note: Lower TTR in Hausa indicates higher lexical repetition,")
    print("  consistent with morphologically rich language patterns.")
    return results


# ── METRICS ──────────────────────────────────

def compute_bleu(refs, hyps):
    return sacrebleu.corpus_bleu(hyps, [refs])


def compute_chrf(refs, hyps, word_order=0):
    return sacrebleu.corpus_chrf(hyps, [refs], word_order=word_order)


def compute_ter(refs, hyps):
    """TER — Translation Edit Rate (sacrebleu). Lower is better."""
    result = sacrebleu.corpus_ter(hyps, [refs])
    return result


def compute_bertscore(refs, hyps):
    """
    BERTScore using multilingual model.
    Requires: pip install bert-score
    Uses bert-base-multilingual-cased which covers Hausa reasonably well.
    """
    try:
        from bert_score import score as bert_score
        print("\n   Computing BERTScore (multilingual-bert) …")
        P, R, F1 = bert_score(
            hyps, refs,
            lang="ha",                          # Hausa
            model_type="bert-base-multilingual-cased",
            verbose=False,
            device="cpu"
        )
        return P.mean().item(), R.mean().item(), F1.mean().item()
    except ImportError:
        print("\n⚠  bert-score not installed. Run:")
        print("    pip install bert-score")
        return None, None, None
    except Exception as e:
        print(f"\n⚠  BERTScore failed: {e}")
        return None, None, None


def compute_africamet(srcs, refs, hyps):
    """AfricoMET — neural MT metric for African languages."""
    try:
        from comet import download_model, load_from_checkpoint
        print("\n   Downloading AfricoMET model …")
        model_path = download_model("masakhane/africamet-mtl")
        model = load_from_checkpoint(model_path)
        data = [{"src": s, "mt": h, "ref": r}
                for s, h, r in zip(srcs, hyps, refs)]
        output = model.predict(data, batch_size=16, gpus=0)
        return output["scores"], output["system_score"]
    except ImportError:
        print("\n⚠  unbabel-comet not installed. Run: pip install unbabel-comet")
        return None, None
    except Exception as e:
        print(f"\n⚠  AfricoMET failed: {e}")
        return None, None


def segment_level_report(refs, hyps, n=5):
    scores = [sacrebleu.sentence_chrf(h, [r]).score for r, h in zip(refs, hyps)]
    indexed = sorted(enumerate(scores), key=lambda x: x[1])
    for label, subset in [("BOTTOM", indexed[:n]), ("TOP", indexed[-n:][::-1])]:
        print(f"\n{'─'*60}")
        print(f"SEGMENT-LEVEL chrF  ({label} {n})")
        print(f"{'─'*60}")
        for i, sc in subset:
            print(f"  [{i:4d}] chrF={sc:.2f}")
            print(f"         REF: {refs[i][:100]}")
            print(f"         HYP: {hyps[i][:100]}")
            print()


# ── MAIN ─────────────────────────────────────

def main(args):
    df = load_data(hf_dataset=args.dataset)

    # ── TTR Analysis (full dataset) ──
    ttr_results = ttr_analysis(df)

    df_eval = filter_pairs(df, REF_COL, args.hyp_col, SRC_COL)
    refs = df_eval[REF_COL].tolist()
    hyps = df_eval[args.hyp_col].tolist()
    srcs = df_eval[SRC_COL].tolist() if SRC_COL in df_eval.columns else None

    print(f"\nEvaluating: REF='{REF_COL}'  HYP='{args.hyp_col}'")
    print("=" * 60)

    # ── BLEU ──
    bleu = compute_bleu(refs, hyps)
    print(f"\n📊 BLEU\n   Score : {bleu.score:.2f}")

    # ── chrF / chrF++ ──
    chrf   = compute_chrf(refs, hyps, word_order=0)
    chrfpp = compute_chrf(refs, hyps, word_order=2)
    print(f"\n📊 chrF\n   chrF   : {chrf.score:.2f}")
    print(f"   chrF++ : {chrfpp.score:.2f}")

    # ── TER ──
    ter = compute_ter(refs, hyps)
    print(f"\n📊 TER\n   Score : {ter.score:.2f}  (lower = better)")

    # ── BERTScore ──
    print(f"\n📊 BERTScore")
    bp, br, bf1 = compute_bertscore(refs, hyps)
    if bf1 is not None:
        print(f"   Precision : {bp:.4f}")
        print(f"   Recall    : {br:.4f}")
        print(f"   F1        : {bf1:.4f}")
    else:
        print("   Not computed (see warning above).")

    # ── AfricoMET ──
    print(f"\n📊 AfricoMET")
    sys_score = None
    if srcs:
        seg_scores, sys_score = compute_africamet(srcs, refs, hyps)
        if sys_score is not None:
            print(f"   System score : {sys_score:.4f}")
        else:
            print("   Not computed (see warning above).")
    else:
        print("   Skipped — source column not available.")

    # ── Summary Table ──
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"  {'Metric':<15} {'Score':>10}  {'Notes'}")
    print(f"  {'─'*50}")
    print(f"  {'BLEU':<15} {bleu.score:>10.2f}  higher is better")
    print(f"  {'chrF':<15} {chrf.score:>10.2f}  higher is better")
    print(f"  {'chrF++':<15} {chrfpp.score:>10.2f}  higher is better")
    print(f"  {'TER':<15} {ter.score:>10.2f}  lower is better")
    if bf1 is not None:
        print(f"  {'BERTScore-F1':<15} {bf1:>10.4f}  higher is better")
    if sys_score is not None:
        print(f"  {'AfricoMET':<15} {sys_score:>10.4f}  higher is better")
    print()

    # ── Segment-level breakdown ──
    if args.segment:
        segment_level_report(refs, hyps, n=args.segment)

    # ── Save to CSV ──
    out_df = pd.DataFrame({
        "source":        srcs if srcs else [""] * len(refs),
        "reference":     refs,
        "hypothesis":    hyps,
        "chrf_segment":  [sacrebleu.sentence_chrf(h, [r]).score for r, h in zip(refs, hyps)],
        "bleu_segment":  [sacrebleu.sentence_bleu(h, [r]).score for r, h in zip(refs, hyps)],
        "ter_segment":   [sacrebleu.sentence_ter(h, [r]).score  for r, h in zip(refs, hyps)],
    })
    out_path = "/content/mt_evaluation_scores.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Segment-level scores saved → {out_path}")


class Args:
    hyp_col = HYP_COL
    dataset  = HF_DATASET
    segment  = 5

main(Args())

Loading from HuggingFace: honourjesus/hausa-pq-speech-validated

────────────────────────────────────────────────────────────
TYPE-TOKEN RATIO (TTR) ANALYSIS
────────────────────────────────────────────────────────────
Mirrors methodology from Eng-PidginBioData (Adelani et al.)

  [Train]  n=539
   TTR (English): 27.6182  [3,336 types / 12,079 tokens]
   TTR (Hausa): 18.6144  [2,942 types / 15,805 tokens]

  [Dev]  n=180
   TTR (English): 28.4862  [1,432 types / 5,027 tokens]
   TTR (Hausa): 20.6714  [1,324 types / 6,405 tokens]

  [Test]  n=180
   TTR (English): 36.4343  [1,547 types / 4,246 tokens]
   TTR (Hausa): 25.8586  [1,408 types / 5,445 tokens]


  Split      Size   TTR(English)   TTR(Hausa)
  ─────────────────────────────────────────────
  Train       539        27.6182      18.6144
  Dev         180        28.4862      20.6714
  Test        180        36.4343      25.8586

  Note: Lower TTR in Hausa indicates higher lexical repetition,
  consistent with morphologically rich 

In [3]:
# ── Run directly in Colab (no argparse needed) ──
class Args:
    hyp_col = HYP_COL
    dataset  = HF_DATASET
    segment  = 5

main(Args())

Loading from HuggingFace: honourjesus/hausa-pq-speech-validated


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/457k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1361 [00:00<?, ? examples/s]


────────────────────────────────────────────────────────────
TYPE-TOKEN RATIO (TTR) ANALYSIS
────────────────────────────────────────────────────────────
Mirrors methodology from Eng-PidginBioData (Adelani et al.)

  [Train]  n=539
   TTR (English): 27.6182  [3,336 types / 12,079 tokens]
   TTR (Hausa): 18.6144  [2,942 types / 15,805 tokens]

  [Dev]  n=180
   TTR (English): 28.4862  [1,432 types / 5,027 tokens]
   TTR (Hausa): 20.6714  [1,324 types / 6,405 tokens]

  [Test]  n=180
   TTR (English): 36.4343  [1,547 types / 4,246 tokens]
   TTR (Hausa): 25.8586  [1,408 types / 5,445 tokens]


  Split      Size   TTR(English)   TTR(Hausa)
  ─────────────────────────────────────────────
  Train       539        27.6182      18.6144
  Dev         180        28.4862      20.6714
  Test        180        36.4343      25.8586

  Note: Lower TTR in Hausa indicates higher lexical repetition,
  consistent with morphologically rich language patterns.

Rows for evaluation: 898  (dropped 463 with 